# Lecture 2 실습: Robot Dynamics & Computed Torque Control

**강의 목표:**
1. `Pinocchio` 라이브러리를 사용하여 로봇의 동역학 모델($M, C, g$)을 계산한다.
2. **Computed Torque Control (CTC)** 기법을 구현하여 로봇이 원하는 궤적을 따라가도록 제어한다.
3. **Forward Dynamics (ABA)**와 **Inverse Dynamics (RNEA)**의 관계를 이해하고, 운동 방정식이 성립하는지 검증한다.

**필요 라이브러리:**
* `pinocchio`: 고속 강체 동역학 라이브러리
* `numpy`, `matplotlib`: 연산 및 시각화

In [ ]:
# 필수 라이브러리 설치 (필요시 주석 해제)
# !pip install pinocchio numpy matplotlib

import pinocchio as pin
import numpy as np
from numpy.linalg import norm
import matplotlib.pyplot as plt
import time

print("Pinocchio version:", pin.__version__)

## 1. Configuration (설정)

시뮬레이션 시간, 제어 이득($K_p, K_d$), 그리고 추종할 궤적(Trajectory)의 파라미터를 설정합니다.
* **PD Gain:** $K_p=400, K_d=40$ (Critical Damping 조건 $K_d \approx 2\sqrt{K_p}$ 고려)
* **Trajectory:** 모든 관절이 `0.5Hz`의 사인파(Sinusoidal) 운동을 하도록 설정합니다.

In [ ]:
class Conf:
    def __init__(self):
        self.dt = 0.001            # Time step [s] (1kHz)
        self.T_SIMULATION = 2.0    # Total simulation time [s]
        
        # PD Control Gains
        self.kp = 40.0            # Proportional gain
        self.kd = 4.0             # Derivative gain
        
        # Reference Trajectory Parameters
        self.freq = 0.5            # Frequency [Hz]
        self.amp = np.pi / 4.0     # Amplitude [rad]
        self.phi = 0.0             # Phase
        self.q0_base = 0.0         # Initial offset

        self.LINE_WIDTH = 60

conf = Conf()
print(f"Simulation setup: dt={conf.dt}s, Total Time={conf.T_SIMULATION}s")

## 2. Controller: Computed Torque Control (CTC)

로봇의 비선형 동역학을 보상하여 선형 시스템처럼 제어하는 기법입니다. 

**제어 법칙 (Control Law):**
$$\tau = M(q)(\ddot{q}_{ref} + K_p e + K_d \dot{e}) + h(q, \dot{q})$$

여기서:
* $e = q_{ref} - q$ (위치 오차)
* $M(q)$: 관성 행렬 (Mass Matrix)
* $h(q, \dot{q})$: 비선형 항 (Coriolis + Centrifugal + Gravity)

이 토크를 로봇에 가하면, 이상적인 경우 폐루프 동역학은 $\ddot{e} + K_d \dot{e} + K_p e = 0$ 이 되어 오차가 0으로 수렴합니다.

In [ ]:
def joint_motion_control(q, v, q_ref, v_ref, dv_ref, kp, kd, nle, M):
    """
    Computed Torque Control Law (Feedback Linearization)
    Args:
        q, v: 현재 관절 위치 및 속도
        q_ref, v_ref, dv_ref: 목표 궤적 (위치, 속도, 가속도)
        nle: 비선형 항 (Non-linear Effects: Coriolis + Gravity)
        M: 관성 행렬 (Mass Matrix)
    """
    # 1. Error calculation
    e = q_ref - q
    de = v_ref - v
    
    # 2. Desired Acceleration (PD Control + Feedforward)
    # dv_cmd = dv_ref + Kp * error + Kd * error_dot
    dv_cmd = dv_ref + kp * e + kd * de
    
    # 3. Inverse Dynamics (Calculate Torque)
    # tau = M(q) * dv_cmd + nle(q, v)
    tau = M @ dv_cmd + nle
    
    return tau

## 3. Simulation Initialization

Pinocchio에 내장된 샘플 6자유도 매니퓰레이터(UR5와 유사)를 로드합니다. 
시뮬레이션 데이터를 저장할 배열들을 초기화합니다.

In [ ]:
# 로봇 모델 로드 (Pinocchio 내장 6-DoF Manipulator)
model = pin.buildSampleModelManipulator()
data = model.createData()

# 초기화
N = int(conf.T_SIMULATION / conf.dt)
robot_nq = model.nq # 관절 개수
robot_nv = model.nv

# 데이터 저장용 배열 (History)
tau_hist    = np.zeros((robot_nv, N))
q_hist      = np.zeros((robot_nq, N))
v_hist      = np.zeros((robot_nv, N))
q_ref_hist  = np.zeros((robot_nq, N))

# 현재 상태 초기화 (Neutral Pose)
q = pin.neutral(model)
v = np.zeros(robot_nv)
conf.q0 = q.copy() # 초기 위치 저장

print(f"Robot Model: {model.name}")
print(f"DoF: {robot_nq}")

## 4. Main Simulation Loop

매 시간 스텝($dt$)마다 다음 과정을 반복합니다:

1.  **Reference Generation:** 목표 궤적($q_{ref}, \dot{q}_{ref}, \ddot{q}_{ref}$) 생성
2.  [cite_start]**Model Computation:** 현재 상태($q, \dot{q}$)에서의 $M(q)$과 $h(q, \dot{q})$ 계산 [cite: 170, 178]
3.  **Controller:** 제어 토크 $\tau$ 계산
4.  **Forward Dynamics:** $\tau$를 입력으로 받아 가속도 $\ddot{q}$ 계산 (**ABA 알고리즘**)
5.  **Integration:** 수치 적분을 통해 다음 스텝의 $q, \dot{q}$ 갱신

In [ ]:
print("Starting simulation...")
start_time = time.time()

t = 0.0
two_pi_f = 2 * np.pi * conf.freq
q_ref = np.zeros(robot_nq)
v_ref = np.zeros(robot_nv)
dv_ref = np.zeros(robot_nv)

for i in range(N):
    # --- A. Reference Trajectory (Sinusoidal) ---
    for j in range(robot_nq):
        q_ref[j]  = conf.q0[j] + conf.amp * np.sin(two_pi_f * t + conf.phi)
        v_ref[j]  = (two_pi_f * conf.amp) * np.cos(two_pi_f * t + conf.phi)
        dv_ref[j] = -(two_pi_f**2 * conf.amp) * np.sin(two_pi_f * t + conf.phi)
    
    # --- B. Dynamics Model Computation ---
    # M(q): Mass Matrix (Composite Rigid Body Algorithm)
    M = pin.crba(model, data, q)
    M = np.tril(M) + np.tril(M, -1).T # Symmetrization for numerical stability
    
    # nle(q, v): Non-linear Effects (Recursive Newton-Euler Algorithm)
    nle = pin.nonLinearEffects(model, data, q, v)
    
    # --- C. Control Law ---
    tau = joint_motion_control(q, v, q_ref, v_ref, dv_ref, conf.kp, conf.kd, nle, M)
    
    # --- D. Forward Dynamics (Physics Simulation) ---
    # a = M^-1 * (tau - nle) 를 구하는 과정 (ABA 알고리즘)
    ddq = pin.aba(model, data, q, v, tau)
    
    # --- E. Integration (Euler Integration) ---
    v += ddq * conf.dt
    q = pin.integrate(model, q, v * conf.dt)
    
    # 데이터 저장
    q_hist[:, i] = q
    v_hist[:, i] = v
    q_ref_hist[:, i] = q_ref
    tau_hist[:, i] = tau
    
    t += conf.dt

print(f"Simulation Finished. Avg Error: {np.mean(norm(q_hist - q_ref_hist, axis=0)):.5f} rad")

## 5. Result Visualization

* **Graph 1:** 각 관절이 목표 궤적(검은 점선)을 잘 따라가는지 확인합니다.
* **Graph 2:** 제어기가 생성한 토크 프로파일을 확인합니다.

In [ ]:
time_array = np.arange(0.0, N * conf.dt, conf.dt)
colors = ['r', 'g', 'b', 'c', 'm', 'y']

# Plot 1: Joint Positions
fig1, ax1 = plt.subplots(3, 2, figsize=(12, 8))
ax1 = ax1.flatten()
for j in range(min(6, robot_nq)): 
    ax1[j].plot(time_array, q_hist[j, :], label=f'q_{j}')
    ax1[j].plot(time_array, q_ref_hist[j, :], 'k--', label=f'Ref_{j}')
    ax1[j].set_ylabel(f'Joint {j} [rad]')
    ax1[j].legend(loc='upper right')
    ax1[j].grid(True)
fig1.suptitle('Joint Positions Tracking', fontsize=16)

# Plot 2: Joint Torques
fig2, ax2 = plt.subplots(3, 2, figsize=(12, 8))
ax2 = ax2.flatten()
for j in range(min(6, robot_nq)):
    ax2[j].plot(time_array, tau_hist[j, :], color=colors[j], label=f'tau_{j}')
    ax2[j].set_ylabel(f'Torque {j} [Nm]')
    ax2[j].grid(True)
fig2.suptitle('Control Input (Torque)', fontsize=16)

plt.tight_layout()
plt.show()

In [ ]:
# 시뮬레이션 중간 지점(t_idx)의 데이터를 추출
t_idx = N // 2
q_check = q_hist[:, t_idx]
v_check = v_hist[:, t_idx]
tau_applied = tau_hist[:, t_idx]

print(f"--- Checking Dynamics at t={t_idx * conf.dt}s ---")

# 1. Forward Dynamics (ABA)로 가속도(a) 계산
# 로봇에게 tau_applied를 주었을 때 발생하는 가속도
a_computed = pin.aba(model, data, q_check, v_check, tau_applied)

# 2. Inverse Dynamics (RNEA)로 토크(tau) 역산
# 그 가속도(a_computed)를 내기 위해 필요한 토크를 역으로 계산
# tau = M(q)a + nle(q, v)
tau_verified = pin.rnea(model, data, q_check, v_check, a_computed)

print(f"Applied Torque (Control):  {tau_applied[:3]} ...")
print(f"RNEA Verified Torque:      {tau_verified[:3]} ...")

# 오차 확인
error = norm(tau_applied - tau_verified)
if error < 1e-10:
    print(f"\n>> [SUCCESS] Dynamics Equation holds! (Error: {error:.2e})")
    print("   검증 완료: Forward Dynamics와 Inverse Dynamics가 수학적으로 일치합니다.")
else:
    print(f"\n>> [FAILURE] Something is wrong. (Error: {error:.2e})")

## 7. [심화] Parameter Estimation using Regressor Matrix

이번에는 로봇의 **동역학 파라미터(질량, 관성 등)를 데이터로부터 추정**해 보겠습니다.
강의 슬라이드 17페이지에 나온 **"Linearity in Parameters"** 성질을 이용합니다.

**식:** $\tau = Y(q, \dot{q}, \ddot{q}) \cdot \pi$

* $Y$: Regressor Matrix (Data Matrix)
* $\pi$: Standard Dynamic Parameters (질량 $m$, 질량중심 $mc$, 관성 $I$ 등 10개의 파라미터 $\times$ 링크 수)

우리는 시뮬레이션을 통해 얻은 데이터($\tau, q, \dot{q}, \ddot{q}$)를 모아서 거대한 행렬 방정식을 만들고, **Linear Regression (Least Squares)**을 수행하여 $\pi$를 구합니다.

In [ ]:
# --- 1. 데이터 준비 (Data Preparation) ---
# 시뮬레이션 전체 구간의 데이터를 사용하여 Regressor를 구성합니다.
# 주의: 데이터가 충분히 "Exciting"해야 정확한 추정이 가능합니다. (Persistent Excitation)

n_samples = N
n_joints = model.nq
n_params = 10 * model.nv # 링크당 10개의 파라미터 (Mass, COM*Mass, Inertia)

# 거대한 Y 행렬과 Tau 벡터를 쌓을 준비
Y_stack = np.zeros((n_samples * n_joints, n_params))
tau_stack = np.zeros(n_samples * n_joints)

print("Building Regressor Matrix Y...")

for k in range(n_samples):
    # 각 시간 스텝의 상태 가져오기
    q_k = q_hist[:, k]
    v_k = v_hist[:, k]
    
    # 가속도는 수치 미분 혹은 ABA의 결과를 사용해야 함.
    # 여기서는 제어 입력에 의한 이상적인 가속도를 사용한다고 가정 (혹은 수치미분)
    if k < n_samples - 1:
        a_k = (v_hist[:, k+1] - v_hist[:, k]) / conf.dt
    else:
        a_k = np.zeros(n_joints)
        
    tau_k = tau_hist[:, k] # 실제 가해진 토크 (Data)
    
    # Pinocchio Regressor 계산
    # Y_k shape: (n_joints, n_params)
    Y_k = pin.computeJointTorqueRegressor(model, data, q_k, v_k, a_k)
    
    # Stacking
    Y_stack[k*n_joints : (k+1)*n_joints, :] = Y_k
    tau_stack[k*n_joints : (k+1)*n_joints] = tau_k

print(f"Regression Problem Size: Y {Y_stack.shape}, tau {tau_stack.shape}")

# --- 2. 선형 회귀 (Linear Regression) ---
# Solve: Y * pi = tau  =>  pi = (Y.T * Y)^-1 * Y.T * tau
# Numpy의 lstsq를 사용하면 수치적으로 안정적으로 풀 수 있습니다.

print("Solving Least Squares...")
start_reg = time.time()

# rcond=None으로 설정하여 rank deficiency 경고를 무시하거나 적절히 처리
pi_est, residuals, rank, s = np.linalg.lstsq(Y_stack, tau_stack, rcond=1e-5)

end_reg = time.time()
print(f"Estimation finished in {end_reg - start_reg:.4f}s")

## 8. 결과 분석 (Estimation Analysis)

추정된 파라미터($\pi_{est}$)와 실제 모델의 파라미터($\pi_{true}$)를 비교합니다.
특히 말단 장치(End-effector) 혹은 특정 링크의 **질량(Mass)**이 정확히 추정되었는지 확인해 봅니다.

* Pinocchio의 파라미터 벡터 순서: 각 링크마다 `[m, mc_x, mc_y, mc_z, I_xx, I_xy, I_yy, I_xz, I_yz, I_zz]` (10개)

In [ ]:
# --- 3. 검증 (Verification) ---

# 실제 모델의 파라미터 벡터 가져오기
# Pinocchio 모델 파일에서 직접 값을 추출하여 벡터로 만듭니다.
pi_true = np.zeros(n_params)

for i in range(model.nv): # 각 링크(Joint)에 대해
    inertia = model.inertias[i+1] # i+1인 이유는 universe(0번) 제외
    
    # Index calculations
    idx = 10 * i
    
    # 1. Mass
    pi_true[idx] = inertia.mass
    # 2. Mass * COM (Lever)
    pi_true[idx+1 : idx+4] = inertia.mass * inertia.lever
    # 3. Rotational Inertia (Standard 3x3 -> 6 unique values)
    # Pinocchio saves inertia in a specific order corresponding to the regressor
    # (Ixx, Ixy, Iyy, Ixz, Iyz, Izz)
    I = inertia.inertia
    pi_true[idx+4] = I[0,0] # Ixx
    pi_true[idx+5] = I[0,1] # Ixy
    pi_true[idx+6] = I[1,1] # Iyy
    pi_true[idx+7] = I[0,2] # Ixz
    pi_true[idx+8] = I[1,2] # Iyz
    pi_true[idx+9] = I[2,2] # Izz

# --- 4. 주요 링크 질량 비교 출력 ---
print("\n" + "".center(conf.LINE_WIDTH,'-'))
print(" Parameter Estimation Results ".center(conf.LINE_WIDTH, '-'))
print("".center(conf.LINE_WIDTH,'-'))

# 마지막 링크 3개에 대해서만 질량 비교
for i in range(max(0, model.nv - 3), model.nv):
    idx = 10 * i
    mass_true = pi_true[idx]
    mass_est = pi_est[idx]
    
    print(f"[Link {i+1}]")
    print(f"  True Mass:      {mass_true:.4f} kg")
    print(f"  Estimated Mass: {mass_est:.4f} kg")
    print(f"  Error:          {abs(mass_true - mass_est):.4f} kg")

# 전체 파라미터에 대한 상대 오차
rel_error = norm(pi_true - pi_est) / (norm(pi_true) + 1e-6)
print(f"\nTotal Parameter Vector Relative Error: {rel_error*100:.2f}%")

if rel_error < 0.05: # 5% 미만이면 성공으로 간주
    print(">> SUCCESS: System Identification works well!")
else:
    print(">> NOTE: Error is high. Try 'Persistent Excitation' trajectory (more complex movement).")

In [ ]:
# --- 1. 설정 변경: 더 복잡하고 긴 궤적 ---
conf.T_SIMULATION = 5.0  # 시간 늘림 (데이터 확보)
N = int(conf.T_SIMULATION / conf.dt)

# "Sum of Sines" 궤적 생성 함수
# 여러 주파수를 섞어서 로봇을 복잡하게 움직임
def get_complex_trajectory(t, n_joints):
    q = np.zeros(n_joints)
    v = np.zeros(n_joints)
    a = np.zeros(n_joints)
    
    # 3개의 주파수 성분 합성 (Fundamental Freq: 0.3Hz)
    freqs = [0.3, 0.7, 1.3] 
    amps =  [0.4, 0.2, 0.1]
    
    for i in range(n_joints):
        # 각 관절마다 위상(Phase)을 다르게 줌
        phase_offset = i * 0.5 
        
        for k in range(len(freqs)):
            omega = 2 * np.pi * freqs[k]
            A = amps[k]
            phi = phase_offset * (k+1)
            
            q[i] += A * np.sin(omega * t + phi)
            v[i] += A * omega * np.cos(omega * t + phi)
            a[i] -= A * omega**2 * np.sin(omega * t + phi)
            
    return q, v, a

# --- 2. 데이터 수집을 위한 시뮬레이션 재실행 ---
print(f"Starting System ID Simulation (T={conf.T_SIMULATION}s)...")
print("Generating 'Rich' Data with Sum-of-Sines Trajectory...")

# 데이터 저장소 초기화
Y_stack = []   # List로 모으고 나중에 vstack (속도 최적화)
tau_stack = []

# 초기 상태
q = np.zeros(robot_nq)
v = np.zeros(robot_nv)
t = 0.0

for i in range(N):
    # A. 복잡한 궤적 생성
    q_ref, v_ref, dv_ref = get_complex_trajectory(t, robot_nq)
    
    # B. 모델 계산 & 제어
    M = pin.crba(model, data, q)
    M = np.tril(M) + np.tril(M, -1).T
    nle = pin.nonLinearEffects(model, data, q, v)
    
    tau = joint_motion_control(q, v, q_ref, v_ref, dv_ref, conf.kp, conf.kd, nle, M)
    
    # C. 시뮬레이션 (ABA + 적분)
    ddq = pin.aba(model, data, q, v, tau)
    v += ddq * conf.dt
    q = pin.integrate(model, q, v * conf.dt)
    
    # D. [중요] Regressor 데이터 수집
    # 가속도는 수치 미분 대신, 제어 입력에 의한 결과인 ddq를 사용 (Ideal case)
    # 실제 실험에선 v를 필터링해서 미분해야 함.
    Y_k = pin.computeJointTorqueRegressor(model, data, q, v, ddq)
    
    # 데이터 쌓기 (Stacking)
    # 초반 과도응답(Transient)은 제외하고 쌓는 것이 좋음 (예: 0.5초 이후)
    if t > 0.5:
        Y_stack.append(Y_k)
        tau_stack.append(tau)
        
    t += conf.dt

# 행렬 형태로 변환
Y_stack = np.vstack(Y_stack)
tau_stack = np.hstack(tau_stack)

print(f"Data Collected. Regression Matrix Size: {Y_stack.shape}")

# --- 3. 회귀 분석 (Least Squares) ---
print("Solving Regression...")
pi_est, _, rank, _ = np.linalg.lstsq(Y_stack, tau_stack, rcond=1e-5)

# --- 4. 결과 검증 (다시 비교) ---
# 실제 파라미터 (True)
pi_true = np.zeros(model.nv * 10)
for i in range(model.nv):
    inert = model.inertias[i+1]
    idx = 10 * i
    pi_true[idx] = inert.mass
    pi_true[idx+1:idx+4] = inert.mass * inert.lever
    I_mat = inert.inertia
    pi_true[idx+4] = I_mat[0,0]
    pi_true[idx+5] = I_mat[0,1]
    pi_true[idx+6] = I_mat[1,1]
    pi_true[idx+7] = I_mat[0,2]
    pi_true[idx+8] = I_mat[1,2]
    pi_true[idx+9] = I_mat[2,2]

# 오차 계산
rel_error = norm(pi_true - pi_est) / norm(pi_true)
print("\n" + "".center(conf.LINE_WIDTH,'='))
print(f" Improved Estimation Result ".center(conf.LINE_WIDTH, '='))
print("".center(conf.LINE_WIDTH,'='))
print(f"Total Relative Error: {rel_error*100:.4f}%")

if rel_error < 1e-4:
    print(">> AMAZING! Parameter Estimation is almost perfect.")
elif rel_error < 0.01:
    print(">> SUCCESS: Very accurate estimation (< 1%).")
else:
    print(">> STILL HIGH ERROR? Check regularization or excitation.")
    
# 주요 링크(마지막) 질량 확인
last_idx = 10 * (model.nv - 1)
print(f"\n[Last Link Mass Verification]")
print(f"  True: {pi_true[last_idx]:.4f}")
print(f"  Est : {pi_est[last_idx]:.4f}")

In [ ]:
# --- 5. 진짜 검증: 토크 재구성 오차 (Torque Reconstruction Error) ---
# 우리가 구한 파라미터(pi_est)가 수식적으로는 완벽한지 확인합니다.
# tau_pred = Y * pi_est

tau_pred = Y_stack @ pi_est
tau_meas = tau_stack

# 토크 오차 (RMS Error)
torque_rmse = np.sqrt(np.mean((tau_meas - tau_pred)**2))
relative_torque_error = norm(tau_meas - tau_pred) / norm(tau_meas)

print("\n" + "".center(conf.LINE_WIDTH,'#'))
print(" Torque Reconstruction Verification ".center(conf.LINE_WIDTH, '#'))
print("".center(conf.LINE_WIDTH,'#'))

print(f"Torque RMS Error:      {torque_rmse:.5f} Nm")
print(f"Relative Torque Error: {relative_torque_error*100:.5f} %")

if relative_torque_error < 0.01: # 1% 미만
    print("\n>> SUCCESS: The Identified Model predicts dynamics PERFECTLY.")
    print("   (Parameter errors are due to 'Base Parameter' redundancy, which is normal.)")
else:
    print("\n>> FAILURE: Even torque prediction is wrong. Check data quality again.")

# --- 6. 시각화: 실제 토크 vs 추정 토크 (일부 구간) ---
# 처음 500개 샘플만 그려서 확인
plt.figure(figsize=(10, 5))
plt.plot(tau_meas[:500], 'k-', label='Measured Torque (Ground Truth)', alpha=0.6)
plt.plot(tau_pred[:500], 'r--', label='Predicted Torque (Identified Model)')
plt.title("Torque Reconstruction (First 500 samples)")
plt.xlabel("Sample Index")
plt.ylabel("Torque [Nm]")
plt.legend()
plt.grid(True)
plt.show()